# MLOps with ClearML — Vehicle Damage Assessment

This notebook implements the full MLOps pipeline aligned with the HLD diagram:

```
Stage 0 → Account Setup & clearml-init
Stage 1 → Dataset Versioning (Feature Store)
Stage 2 → Experiment Tracking (Orchestrated Experiment)
Stage 3 → Model Evaluation & Registry (Model Registry)
Stage 4 → Automated Pipeline (Full CI/CD)
Stage 5 → Performance Monitoring (Trigger)
```

**Prerequisites:** Sign up for a free account at https://app.clear.ml before running Stage 0.

---
## Stage 0 — Account Setup
Run this cell once to connect your local machine to ClearML.
It will open a browser and ask you to paste your API credentials.

In [ ]:
# Install ClearML if not already installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'clearml', '-q'])
print('ClearML installed.')

In [ ]:
# Run clearml-init to paste your API credentials
# Go to https://app.clear.ml → Settings → Workspace → Create new credentials
# Then paste the block it gives you here:
from clearml.config import put_config_file
import clearml

print(f'ClearML version: {clearml.__version__}')
print()
print('To configure credentials, run in your terminal:')
print('  clearml-init')
print()
print('OR paste the credentials block from https://app.clear.ml below:')

In [ ]:
# ─── PASTE YOUR CREDENTIALS HERE ───────────────────────────────────────────
# Get these from: https://app.clear.ml → Settings → Workspace → Create credentials
# It will give you a block that looks like:
#
#   api {
#     web_server: https://app.clear.ml
#     api_server: https://api.clear.ml
#     files_server: https://files.clear.ml
#     credentials {
#       access_key: "YOUR_ACCESS_KEY"
#       secret_key: "YOUR_SECRET_KEY"
#     }
#   }
#
# Paste the entire block as a string below:

CLEARML_CREDENTIALS = """
api {
  web_server: https://app.clear.ml
  api_server: https://api.clear.ml
  files_server: https://files.clear.ml
  credentials {
    "access_key" = "PASTE_YOUR_ACCESS_KEY"
    "secret_key"  = "PASTE_YOUR_SECRET_KEY"
  }
}
"""

import os
clearml_conf_path = os.path.expanduser('~/clearml.conf')
with open(clearml_conf_path, 'w') as f:
    f.write(CLEARML_CREDENTIALS)

print(f'Credentials written to {clearml_conf_path}')
print('Now restart the kernel and run Stage 1.')

---
## Stage 1 — Dataset Versioning
> **HLD Stage:** Data Analysis → Feature Store

Register your CarDD dataset in ClearML so every training run is reproducible.

In [ ]:
from clearml import Dataset
import os

# Adjust this path to where your CarDD data lives
CARDD_ROOT = '../data/CarDD/CarDD_COCO'

# Verify the data exists
for folder in ['train2017', 'val2017', 'test2017', 'annotations']:
    path = os.path.join(CARDD_ROOT, folder)
    exists = os.path.isdir(path)
    count = len(os.listdir(path)) if exists else 0
    print(f'  {folder}: {"✓" if exists else "✗"} ({count} files)')

In [ ]:
# Fetch the existing CarDD_YOLO dataset from your workspace
dataset = Dataset.get(dataset_id='0a5ce277be174c47bd75b3ef202a3b50')
dataset_path = dataset.get_local_copy()

print(f'\n✓ Dataset retrieved!')
print(f'  Dataset ID : {dataset.id}')
print(f'  Name       : {dataset.name}')
print(f'  Local path : {dataset_path}')


---
## Stage 2 — Experiment Tracking (Orchestrated Experiment)
> **HLD Stage:** Orchestrated Experiment → Source Code → Source Repository

Run a tracked training experiment. ClearML auto-logs:
- All hyperparameters
- Loss curves, mAP, precision, recall per epoch
- The trained `.pt` model
- GPU usage and system metrics
- Git commit hash + code diff

In [ ]:
from clearml import Task, Dataset, OutputModel

# ─── Configuration ──────────────────────────────────────────────────────────
# Change MODEL_NAME to switch between: 'yolo11m', 'yolov8m'
MODEL_NAME       = 'yolo11m'
DATASET_ID       = '0a5ce277be174c47bd75b3ef202a3b50'
EPOCHS           = 100
BATCH_SIZE       = 16
IMAGE_SIZE       = 640
LR               = 0.001
CONF_THRESHOLD   = 0.25

In [ ]:
from ultralytics import YOLO

# ── Init ClearML Task — this captures everything from this point on ──────────
task = Task.init(
    project_name='Intelligent Vehicle Damage Assessment',
    task_name=f'{MODEL_NAME}-Training',
    task_type=Task.TaskTypes.training,
    reuse_last_task_id=False,
)

# ── Connect hyperparameters — editable from the ClearML UI ──────────────────
params = task.connect({
    'model_name':      MODEL_NAME,
    'dataset_id':      DATASET_ID,
    'epochs':          EPOCHS,
    'batch_size':      BATCH_SIZE,
    'image_size':      IMAGE_SIZE,
    'lr':              LR,
    'conf_threshold':  CONF_THRESHOLD,
})

# ── Also connect the training config file ───────────────────────────────────
task.connect_configuration('../configs/training_config.yaml')

print(f'✓ Task initialised: {task.name}')
print(f'  View at: https://app.clear.ml')

In [ ]:
# ── Load dataset from ClearML (reproducible, versioned) ──────────
dataset = Dataset.get(dataset_id=params['dataset_id'])
dataset_path = dataset.get_local_copy()
print(f'✓ Dataset loaded from: {dataset_path}')


In [ ]:
# ── Train model — ClearML auto-intercepts Ultralytics output ────────────────
model = YOLO(f"{params['model_name']}.pt")

results = model.train(
    data=f'{dataset_path}/cardd.yaml',
    epochs=params['epochs'],
    batch=params['batch_size'],
    imgsz=params['image_size'],
    lr0=params['lr'],
    project='runs/clearml_train',
    name=params['model_name'],
    exist_ok=True,
) 

print('\n✓ Training complete!')

In [ ]:
# ── Log final metrics explicitly ─────────────────────────────────────────────
logger = task.get_logger()

metrics = {
    'mAP@0.5':    results.results_dict.get('metrics/mAP50', 0),
    'mAP@0.5:95': results.results_dict.get('metrics/mAP50-95', 0),
    'Precision':  results.results_dict.get('metrics/precision', 0),
    'Recall':     results.results_dict.get('metrics/recall', 0),
}

for metric_name, value in metrics.items():
    logger.report_scalar('Final Metrics', metric_name, value=value, iteration=0)
    print(f'  {metric_name}: {value:.4f}')

# ── Register model weights in ClearML Model Registry ────────────────────────
output_model = OutputModel(
    task=task,
    label_enumeration={
        'dent': 0, 'scratch': 1, 'crack': 2,
        'glass shatter': 3, 'lamp broken': 4, 'tire flat': 5,
    }
)

best_weights = f"runs/clearml_train/{params['model_name']}/weights/best.pt"
output_model.update_weights(weights_filename=best_weights, auto_delete_file=False)
output_model.update_design(config_dict=params)

task.close()
print(f'\n✓ Model registered in ClearML Model Registry!')
print(f'  Task ID (save this): {task.id}')

---
## Stage 3 — Model Evaluation & Registry
> **HLD Stage:** Model Evaluation → Model Validation → Model Registry

Compare all trained models side by side and promote the best one.

In [ ]:
from clearml import Task, Model
from ultralytics import YOLO

# ── Paste the Task IDs from your training runs above ────────────────────────
# Example: MODELS_TO_COMPARE = {'YOLO11m': 'abc123', 'YOLOv8m': 'def456'}
MODELS_TO_COMPARE = {
    'YOLO11m': 'PASTE_TASK_ID_FROM_STAGE_2',
    # 'YOLOv8m': 'PASTE_TASK_ID_FROM_YOLOV8_RUN',
}

eval_task = Task.init(
    project_name='Intelligent Vehicle Damage Assessment',
    task_name='Model-Evaluation-Comparison',
    task_type=Task.TaskTypes.testing,
)

logger = eval_task.get_logger()
comparison = {}

for model_name, task_id in MODELS_TO_COMPARE.items():
    print(f'Evaluating {model_name}...')
    
    # Load model from ClearML registry
    registered_model = Model(task=Task.get_task(task_id=task_id))
    weights_path = registered_model.get_local_copy()
    
    # Run validation
    yolo = YOLO(weights_path)
    val_metrics = yolo.val(data='../data/CarDD/cardd.yaml', conf=0.25)
    
    comparison[model_name] = {
        'mAP@0.5':    round(val_metrics.box.map50, 4),
        'mAP@0.5:95': round(val_metrics.box.map,   4),
        'Precision':  round(val_metrics.box.mp,     4),
        'Recall':     round(val_metrics.box.mr,     4),
    }
    
    # Log to ClearML
    for metric, value in comparison[model_name].items():
        logger.report_scalar(f'Comparison/{metric}', model_name, value, 0)

# ── Report comparison table ──────────────────────────────────────────────────
logger.report_table('Model Comparison', 'Results', iteration=0, table_plot=comparison)

# Select best
best_model = max(comparison, key=lambda m: comparison[m]['mAP@0.5:95'])
print(f'\n✓ Best model: {best_model}')
for k, v in comparison[best_model].items():
    print(f'  {k}: {v}')

eval_task.close()

---
## Stage 4 — Automated Pipeline
> **HLD Stage:** Automated Pipeline (Data extraction → Validation → Preparation → Training → Evaluation → Validation)

This stage wires all the above into a single end-to-end pipeline that can be triggered automatically.

In [ ]:
from clearml.automation.controller import PipelineDecorator

# ── Pipeline step 1: Validate dataset ───────────────────────────────────────
@PipelineDecorator.component(
    return_values=['dataset_id'],
    cache=True,
    packages=['clearml'],
)
def step_validate_dataset(dataset_id: str):
    from clearml import Dataset
    import os

    dataset = Dataset.get(dataset_id=dataset_id)
    path = dataset.get_local_copy()

    required = ['train2017', 'val2017', 'test2017', 'annotations']
    for folder in required:
        assert os.path.isdir(os.path.join(path, folder)), f'Missing folder: {folder}'

    count = len(os.listdir(os.path.join(path, 'train2017')))
    print(f'✓ Dataset valid — {count} training images')
    return dataset.id


# ── Pipeline step 2: Train model ─────────────────────────────────────────────
@PipelineDecorator.component(
    return_values=['model_task_id'],
    cache=False,
    packages=['clearml', 'ultralytics'],
)
def step_train(dataset_id: str, model_name: str, epochs: int):
    from clearml import Task, Dataset, OutputModel
    from ultralytics import YOLO

    task = Task.current_task()
    dataset = Dataset.get(dataset_id=dataset_id)
    dataset_path = dataset.get_local_copy()

    model = YOLO(f'{model_name}.pt')
    results = model.train(
        data=f'{dataset_path}/cardd.yaml',
        epochs=epochs,
        imgsz=640,
        batch=16,
    )

    output_model = OutputModel(task=task)
    output_model.update_weights(f'runs/train/weights/best.pt')
    return task.id


# ── Pipeline step 3: Validate model quality ───────────────────────────────────
@PipelineDecorator.component(
    return_values=['passed', 'map50'],
    cache=False,
)
def step_validate_model(model_task_id: str, min_map: float = 0.50):
    from clearml import Task, Model
    from ultralytics import YOLO

    registered = Model(task=Task.get_task(task_id=model_task_id))
    yolo = YOLO(registered.get_local_copy())
    metrics = yolo.val()

    map50 = metrics.box.map50
    passed = map50 >= min_map
    print(f'mAP@0.5={map50:.4f} | Threshold={min_map} | {"PASS ✓" if passed else "FAIL ✗"}')
    return passed, map50


# ── Full pipeline controller ──────────────────────────────────────────────────
@PipelineDecorator.pipeline(
    name='Vehicle-Damage-Full-Pipeline',
    project='Intelligent Vehicle Damage Assessment',
    version='1.0',
)
def run_pipeline(
    dataset_id: str = '0a5ce277be174c47bd75b3ef202a3b50',
    epochs: int = 100,
    model_name: str = 'yolo11m',
    min_map: float = 0.50,
):
    dataset_id    = step_validate_dataset(dataset_id)
    model_task_id = step_train(dataset_id, model_name, epochs)
    passed, map50 = step_validate_model(model_task_id, min_map)

    if passed:
        print(f'\n✓ Pipeline complete. Model ready for serving (mAP@0.5={map50:.4f})')
    else:
        print(f'\n✗ Model failed validation. Not deploying (mAP@0.5={map50:.4f} < {min_map})')


# ── Run locally for testing (no agent needed) ─────────────────────────────────
PipelineDecorator.run_locally()

run_pipeline(
    dataset_id='0a5ce277be174c47bd75b3ef202a3b50',
    epochs=1,            # Set to 100 for real training
    model_name='yolo11m',
    min_map=0.50,
)
print('Done!')


---
## Stage 5 — Performance Monitoring
> **HLD Stage:** Performance Monitoring → Trigger

This stage logs every real inference from FastAPI to ClearML so you can detect when the model degrades in production and automatically trigger retraining.

In [ ]:
# This code block shows what to ADD to backend/app/routers/damage.py
# It is shown here for reference — do not run this cell directly in the notebook.

MONITORING_CODE = '''
# Add to the top of backend/app/routers/damage.py:

from clearml import Task

_monitor_task = None
_request_count = 0

def get_monitor():
    global _monitor_task
    if _monitor_task is None:
        _monitor_task = Task.init(
            project_name="Intelligent Vehicle Damage Assessment",
            task_name="Production-Monitoring",
            task_type=Task.TaskTypes.monitor,
            reuse_last_task_id=True,
        )
    return _monitor_task

# Inside your /damage/predict endpoint, after inference:
global _request_count
_request_count += 1
monitor = get_monitor()
logger = monitor.get_logger()
logger.report_scalar("Production", "inference_ms",    result.inference_time_ms, _request_count)
logger.report_scalar("Production", "num_detections",  result.num_detections,    _request_count)
logger.report_scalar("Production", "avg_confidence",  avg_conf,                 _request_count)
'''

print('── Code to add to backend/app/routers/damage.py ──')
print(MONITORING_CODE)

In [ ]:
# Set up auto-retrain trigger (run once after monitoring is active)
# This watches the production metrics and fires the pipeline if quality drops

from clearml.automation import TriggerScheduler

# ── Fill in your IDs ─────────────────────────────────────────────────────────
MONITORING_TASK_ID = 'PASTE_PRODUCTION_MONITORING_TASK_ID'
PIPELINE_ID        = 'PASTE_PIPELINE_TASK_ID'

scheduler = TriggerScheduler()

scheduler.add_task_trigger(
    schedule_task_id=MONITORING_TASK_ID,
    trigger_on_metric='Production/avg_confidence',
    trigger_on_sign=-1,         # Triggers when value DECREASES
    trigger_threshold=0.05,     # by more than 5%
    schedule_pipeline_id=PIPELINE_ID,
)

print('✓ Trigger scheduler configured.')
print('  The pipeline will auto-run when avg_confidence drops more than 5%.')
# scheduler.start()  # Uncomment to activate

---
## Summary

| Stage | HLD Component | ClearML Feature | Status |
|-------|--------------|-----------------|--------|
| 0 | — | `clearml-init` credentials | Run once |
| 1 | Feature Store | `Dataset.create()` | Versions your CarDD data |
| 2 | Orchestrated Experiment | `Task.init()` | Auto-logs all training runs |
| 3 | Model Registry | `OutputModel` | Stores `.pt` with metrics |
| 4 | Automated Pipeline | `PipelineDecorator` | End-to-end CI/CD |
| 5 | Performance Monitoring | `Task(monitor)` + Trigger | Auto-retrain on drift |

**Dashboard:** https://app.clear.ml → Your Project: `Intelligent Vehicle Damage Assessment`